In [ ]:
import os
from typing import List,TypedDict,Annotated
from langgraph.graph import StateGraph,START,END,MessagesState, add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,ToolMessage,AIMessage      
from langchain_openai import AzureChatOpenAI,AzureOpenAIEmbeddings
from dotenv import load_dotenv,find_dotenv
from IPython.display import Image

load_dotenv(find_dotenv(),override=True)
endpoint = os.getenv("ENDPOINT_URL")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version="2025-01-01-preview"

class AgentState(TypedDict):
    messages: Annotated[List,add_messages]

@tool
def currentTemperature(country:str) -> int :
    '''current temperature of the country'''
    if country.startswith("i"):
         return 10
    elif country.startswith("u"):
        return 20
    else:
        return 30

tools=[currentTemperature]
llm = (AzureChatOpenAI(azure_endpoint=endpoint,api_key=subscription_key,api_version=api_version)).bind_tools(tools)
conn=sqlite3.connect("checkpoint.db",check_same_thread=False)
memory=SqliteSaver(conn)

def call_model(state:AgentState):
    res=llm.invoke(state["messages"]) 
    return {"messages": [ res.to_json() ]}

def should_continue(state:AgentState):
    messages=state["messages"]
    last_message=messages[-1]
    if last_message.tool_calls:
        return "tools"
    else:
        return END

graph=StateGraph(AgentState)
tool_node=ToolNode(tools)

graph.add_node("agent",call_model)
graph.add_node("tools",tool_node)

graph.add_edge(START,"agent")
graph.add_conditional_edges("agent",should_continue)
graph.add_edge("tools","agent")

app=graph.compile(checkpointer=memory)
# 4. Use a thread_id to identify the conversation
config = {"configurable": {"thread_id": "1"}}

results = app.invoke({"messages": [HumanMessage(content="Hi, I'm Bob", role="user")]}, config)